# Fase de dados — CodeXGLUE

Normaliza e analisa PHP, Python e JavaScript. SQL permanece em pipeline curado separado. Esta etapa não treina o modelo.

In [ ]:
from google.colab import files
from pathlib import Path
import json
import os
import shutil

uploaded = files.upload()  # selecione Celx-colab-v8.zip
archives = [name for name in uploaded if name.lower().endswith('.zip')]
assert archives, 'Envie Celx-colab-v8.zip.'
project_root = Path('/content/legacy-doc-project')
if project_root.exists():
    shutil.rmtree(project_root)
project_root.mkdir(parents=True)
shutil.unpack_archive(f'/content/{archives[0]}', project_root)
assert (project_root / 'configs/default.yaml').exists()
os.environ['PYTHONPATH'] = str(project_root)
print('Projeto:', project_root)

In [ ]:
%cd /content/legacy-doc-project
%pip install -q "datasets>=3.0" "transformers>=4.51" "PyYAML>=6.0"
%pip install -q -e .
print('Ambiente preparado.')

## Normalização

Baixa as configurações Python, PHP e JavaScript do CodeXGLUE. As divisões são deduplicadas por hash, reservando teste e validação antes do treino.

In [ ]:
!python scripts/prepare_dataset.py --config configs/default.yaml

In [ ]:
!python scripts/analyze_dataset.py --config configs/default.yaml
from IPython.display import Markdown, display
analysis_md = Path('evaluation/dataset_analysis/analysis.md')
display(Markdown(analysis_md.read_text(encoding='utf-8')))

## Benchmark expandido

Seleciona 100 exemplos únicos por linguagem: Python, PHP e JavaScript do CodeXGLUE; SQL do Spider. Total: 400 casos.

In [ ]:
!python scripts/build_benchmark.py
benchmark_manifest = json.loads(
    Path('dataset/benchmark/expanded_100.manifest.json').read_text(encoding='utf-8')
)
display(benchmark_manifest)

## Persistência no Google Drive

Salva os dados normalizados e o relatório para que não sejam perdidos quando a sessão terminar.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
destination = Path('/content/drive/MyDrive/Colab Notebooks/Fine-tuning/data_phase')
destination.mkdir(parents=True, exist_ok=True)
shutil.copytree(
    Path('dataset/processed/codexglue'),
    destination / 'codexglue',
    dirs_exist_ok=True,
)
shutil.copytree(
    Path('evaluation/dataset_analysis'),
    destination / 'analysis',
    dirs_exist_ok=True,
)
shutil.copy2(
    Path('dataset/benchmark/expanded_100.jsonl'),
    destination / 'expanded_100.jsonl',
)
shutil.copy2(
    Path('dataset/benchmark/expanded_100.manifest.json'),
    destination / 'expanded_100.manifest.json',
)
print('Artefatos salvos em:', destination)

In [ ]:
files.download('evaluation/dataset_analysis/analysis.md')
files.download('evaluation/dataset_analysis/analysis.json')
files.download('dataset/processed/codexglue/manifest.json')
files.download('dataset/benchmark/expanded_100.jsonl')
files.download('dataset/benchmark/expanded_100.manifest.json')

## Próxima atividade

Os dados normalizados ainda não estão prontos para Fine-Tuning. O próximo trabalho é selecionar exemplos, gerar documentação estruturada em português e submetê-la à revisão humana antes de construir o dataset SFT.